# Agent Swarms: Multi-Agent Collaboration with Anthropic

A single LLM agent can only approach a problem from one perspective at a time. Agent swarms spawn multiple specialized agents in parallel, each exploring the problem differently, then aggregate their findings. This mirrors how research teams work: specialists investigate independently, then synthesize results.

This notebook uses the **Anthropic Python SDK** with `asyncio` for parallel agent execution.

Swarms excel at:
- **Ambiguous queries** where multiple interpretations are valid
- **Research tasks** requiring broad coverage
- **Creative generation** where diversity matters
- **Fact verification** through independent corroboration

In [ ]:
import os
import json
import asyncio
from getpass import getpass
from dataclasses import dataclass
from collections import Counter
from anthropic import Anthropic, AsyncAnthropic

# API Key Setup
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

# Sync client for simple operations
client = Anthropic()

# Async client for parallel operations
async_client = AsyncAnthropic()

MODEL = "claude-sonnet-4-20250514"

## Part 1: Basic Fan-Out Pattern

The simplest swarm: send the same query to multiple agents with different personas, collect all responses in parallel using `asyncio.gather()`.

In [ ]:
# Define agent personas for a research task
RESEARCH_PERSONAS = [
    {
        "name": "Technical Expert",
        "system": "You are a technical expert. Focus on implementation details, code, algorithms, and engineering tradeoffs. Be specific and precise."
    },
    {
        "name": "Business Analyst", 
        "system": "You are a business analyst. Focus on costs, ROI, market trends, and practical business applications. Think about the bottom line."
    },
    {
        "name": "Skeptical Critic",
        "system": "You are a skeptical critic. Focus on limitations, risks, failure modes, and what could go wrong. Challenge assumptions."
    },
    {
        "name": "Creative Innovator",
        "system": "You are a creative innovator. Focus on novel applications, future possibilities, and unconventional uses. Think outside the box."
    }
]

In [ ]:
async def fan_out_query(query: str, personas: list[dict]) -> list[dict]:
    """Send query to multiple agents in parallel using asyncio.gather()."""
    
    async def query_agent(persona: dict) -> dict:
        """Query a single agent with its persona."""
        response = await async_client.messages.create(
            model=MODEL,
            max_tokens=1024,
            system=persona["system"],
            messages=[{"role": "user", "content": query}]
        )
        return {
            "agent": persona["name"],
            "response": response.content[0].text
        }
    
    # Run all agents in parallel
    tasks = [query_agent(p) for p in personas]
    results = await asyncio.gather(*tasks)
    return results

# Test fan-out
query = "What are the implications of using LLMs for code generation in production?"
results = await fan_out_query(query, RESEARCH_PERSONAS)

print(f"Query: {query}\n")
for r in results:
    print(f"=== {r['agent']} ===")
    print(r['response'][:500] + "...\n")

## Part 2: Aggregation Strategies

Raw outputs from multiple agents are hard to use. We need to aggregate them into a coherent response.

### Strategy 1: Editor Agent

An "editor" agent synthesizes all responses into a unified answer.

In [ ]:
def aggregate_with_editor(query: str, agent_responses: list[dict]) -> str:
    """Use an editor agent to synthesize multiple perspectives."""
    
    # Format responses for the editor
    formatted_responses = "\n\n".join([
        f"### {r['agent']}\n{r['response']}" 
        for r in agent_responses
    ])
    
    editor_prompt = f"""You are an editor synthesizing research from multiple analysts.

Original Question: {query}

Analyst Responses:
{formatted_responses}

Create a unified, well-structured response that:
1. Identifies key points where analysts agree
2. Notes important disagreements or different perspectives
3. Synthesizes into actionable insights
4. Is concise but comprehensive

Do not simply concatenate the responses. Create a new, integrated analysis."""
    
    response = client.messages.create(
        model=MODEL,
        max_tokens=2048,
        messages=[{"role": "user", "content": editor_prompt}]
    )
    
    return response.content[0].text

# Test editor aggregation
synthesized = aggregate_with_editor(query, results)
print("=== SYNTHESIZED RESPONSE ===")
print(synthesized)

### Strategy 2: Voting / Consensus

For questions with discrete answers, agents can vote. Useful for classification, fact-checking, or yes/no decisions.

In [ ]:
async def voting_swarm(question: str, options: list[str], n_agents: int = 5) -> dict:
    """Multiple agents vote on a question with discrete options."""
    
    options_str = ", ".join(options)
    
    async def get_vote(agent_id: int) -> str:
        prompt = f"""Question: {question}

Options: {options_str}

Think through this carefully, then respond with ONLY one of the options listed above.
Your answer:"""
        
        response = await async_client.messages.create(
            model=MODEL,
            max_tokens=100,
            messages=[{"role": "user", "content": prompt}]
        )
        
        # Extract the vote (should be one of the options)
        vote = response.content[0].text.strip()
        # Normalize to match options
        for opt in options:
            if opt.lower() in vote.lower():
                return opt
        return vote  # Return raw if no match
    
    # Collect votes in parallel
    tasks = [get_vote(i) for i in range(n_agents)]
    votes = await asyncio.gather(*tasks)
    
    # Count votes
    vote_counts = Counter(votes)
    winner = vote_counts.most_common(1)[0]
    
    return {
        "votes": votes,
        "counts": dict(vote_counts),
        "winner": winner[0],
        "confidence": winner[1] / len(votes)
    }

# Test voting
result = await voting_swarm(
    question="Is Python or JavaScript better for building a REST API backend?",
    options=["Python", "JavaScript", "Both are equally good", "Depends on the use case"],
    n_agents=7
)

print(f"Individual votes: {result['votes']}")
print(f"Vote counts: {result['counts']}")
print(f"Winner: {result['winner']} (confidence: {result['confidence']:.0%})")

### Strategy 3: Confidence-Weighted Aggregation

Ask agents to rate their confidence. Weight their contributions accordingly.

In [ ]:
import re

async def confidence_weighted_query(query: str, n_agents: int = 5) -> dict:
    """Query multiple agents and weight by confidence."""
    
    async def get_confident_response(agent_id: int) -> dict:
        prompt = f"""{query}

Provide your response in this exact format:
ANSWER: [Your answer here]
CONFIDENCE: [1-10]
REASONING: [Brief explanation]"""
        
        response = await async_client.messages.create(
            model=MODEL,
            max_tokens=500,
            messages=[{"role": "user", "content": prompt}]
        )
        
        text = response.content[0].text
        
        # Parse the response
        answer_match = re.search(r'ANSWER:\s*(.+?)(?=CONFIDENCE:|$)', text, re.DOTALL)
        confidence_match = re.search(r'CONFIDENCE:\s*(\d+)', text)
        reasoning_match = re.search(r'REASONING:\s*(.+?)$', text, re.DOTALL)
        
        return {
            "answer": answer_match.group(1).strip() if answer_match else text[:200],
            "confidence": int(confidence_match.group(1)) if confidence_match else 5,
            "reasoning": reasoning_match.group(1).strip() if reasoning_match else ""
        }
    
    # Collect responses in parallel
    tasks = [get_confident_response(i) for i in range(n_agents)]
    responses = await asyncio.gather(*tasks)
    
    # Weight by confidence
    total_confidence = sum(r["confidence"] for r in responses)
    weighted_responses = [
        {
            "answer": r["answer"],
            "confidence": r["confidence"],
            "weight": r["confidence"] / total_confidence if total_confidence > 0 else 1/len(responses),
            "reasoning": r["reasoning"]
        }
        for r in responses
    ]
    
    # Sort by confidence
    weighted_responses.sort(key=lambda x: x["confidence"], reverse=True)
    
    return {
        "responses": weighted_responses,
        "top_answer": weighted_responses[0]["answer"],
        "avg_confidence": total_confidence / n_agents
    }

# Test
result = await confidence_weighted_query(
    "What is the most important factor when choosing a database for a startup?",
    n_agents=4
)

print(f"Average confidence: {result['avg_confidence']:.1f}/10\n")
for r in result["responses"]:
    print(f"[Confidence: {r['confidence']}/10, Weight: {r['weight']:.0%}]")
    print(f"Answer: {r['answer'][:200]}...")
    print()

## Part 3: Research Swarm

A complete research swarm with specialized agents and multi-stage processing.

In [ ]:
@dataclass
class ResearchSwarm:
    """Multi-agent research swarm with specialized roles using Anthropic."""
    
    async def research(self, topic: str) -> dict:
        """Run a complete research cycle on a topic."""
        
        # Stage 1: Generate research questions
        print("Stage 1: Generating research questions...")
        questions = await self._generate_questions(topic)
        
        # Stage 2: Research each question in parallel
        print(f"Stage 2: Researching {len(questions)} questions in parallel...")
        findings = await self._parallel_research(questions)
        
        # Stage 3: Synthesize findings
        print("Stage 3: Synthesizing findings...")
        synthesis = await self._synthesize(topic, findings)
        
        # Stage 4: Critical review
        print("Stage 4: Critical review...")
        final_report = await self._critical_review(topic, synthesis)
        
        return {
            "topic": topic,
            "questions": questions,
            "findings": findings,
            "synthesis": synthesis,
            "final_report": final_report
        }
    
    async def _generate_questions(self, topic: str) -> list[str]:
        """Generate research sub-questions."""
        response = await async_client.messages.create(
            model=MODEL,
            max_tokens=500,
            messages=[{
                "role": "user", 
                "content": f"""Generate 4 specific research questions about: {topic}

Questions should cover:
1. Technical/how it works
2. Benefits and advantages
3. Limitations and challenges
4. Practical applications

Return as a JSON array of strings only, no other text."""
            }]
        )
        # Parse JSON from response
        text = response.content[0].text
        # Find JSON array in response
        start = text.find('[')
        end = text.rfind(']') + 1
        if start != -1 and end > start:
            return json.loads(text[start:end])
        return json.loads(text)
    
    async def _parallel_research(self, questions: list[str]) -> list[dict]:
        """Research each question with a specialized agent in parallel."""
        
        async def research_question(q: str) -> dict:
            response = await async_client.messages.create(
                model=MODEL,
                max_tokens=500,
                messages=[{
                    "role": "user",
                    "content": f"""Research question: {q}

Provide a detailed, factual response. Include specific examples where relevant.
Keep response under 300 words."""
                }]
            )
            return {"question": q, "finding": response.content[0].text}
        
        tasks = [research_question(q) for q in questions]
        return await asyncio.gather(*tasks)
    
    async def _synthesize(self, topic: str, findings: list[dict]) -> str:
        """Synthesize findings into a coherent narrative."""
        findings_text = "\n\n".join([
            f"Q: {f['question']}\nA: {f['finding']}" 
            for f in findings
        ])
        
        response = await async_client.messages.create(
            model=MODEL,
            max_tokens=1500,
            messages=[{
                "role": "user",
                "content": f"""Topic: {topic}

Research Findings:
{findings_text}

Synthesize these findings into a coherent, well-structured analysis.
Identify patterns, connections, and key insights across the findings."""
            }]
        )
        return response.content[0].text
    
    async def _critical_review(self, topic: str, synthesis: str) -> str:
        """Critical review and final polish."""
        response = await async_client.messages.create(
            model=MODEL,
            max_tokens=2000,
            system="You are a critical reviewer. Identify gaps, suggest improvements, and add nuance.",
            messages=[{
                "role": "user",
                "content": f"""Topic: {topic}

Draft Analysis:
{synthesis}

Review this analysis critically. Add:
1. Any important missing perspectives
2. Caveats or limitations
3. A brief executive summary at the top

Return the improved final report."""
            }]
        )
        return response.content[0].text

In [ ]:
# Run the research swarm
swarm = ResearchSwarm()
result = await swarm.research("Vector databases for AI applications")

print("\n" + "="*60)
print("RESEARCH QUESTIONS:")
print("="*60)
for i, q in enumerate(result["questions"], 1):
    print(f"{i}. {q}")

print("\n" + "="*60)
print("FINAL REPORT:")
print("="*60)
print(result["final_report"])

## When to Use Swarms

| Use Case | Single Agent | Swarm | Why |
|----------|-------------|-------|-----|
| Simple Q&A | ✓ | | Overkill |
| Research/analysis | | ✓ | Multiple perspectives valuable |
| Classification | | ✓ | Voting improves accuracy |
| Creative writing | | ✓ | Diversity matters |
| Code generation | ✓ | | Consistency more important |
| Fact verification | | ✓ | Independent corroboration |
| Cost-sensitive | ✓ | | Swarms multiply API costs |

### Cost Considerations

A 5-agent swarm costs 5x a single agent call. However, because agents run in parallel with `asyncio.gather()`, latency is roughly the same as a single agent call.

**Use swarms when**: The value of multiple perspectives exceeds the additional cost.

---

## Exercises

### Exercise 1: Debate Swarm

Create a swarm where agents take opposing positions and debate. An arbiter agent decides the winner based on argument quality.

In [ ]:
async def debate_swarm(topic: str, position_a: str, position_b: str, rounds: int = 2):
    """Two agents debate, arbiter decides winner."""
    # Your implementation here
    pass

### Exercise 2: Self-Improving Swarm

Create a swarm where agents critique each other's outputs, then refine their answers based on peer feedback.

In [ ]:
async def peer_review_swarm(question: str, n_agents: int = 3):
    """Agents answer, critique each other, then revise."""
    # Your implementation here
    pass

### Exercise 3: Adaptive Swarm Size

Create a swarm that starts with 2 agents, adds more if they disagree significantly, and stops when consensus is reached.

In [ ]:
async def adaptive_swarm(question: str, max_agents: int = 7):
    """Dynamically add agents until consensus."""
    # Your implementation here
    pass

---

## Summary

- **Fan-out pattern**: Send same query to multiple specialized agents in parallel using `asyncio.gather()`
- **Editor aggregation**: Use a synthesis agent to combine perspectives
- **Voting**: For discrete answers, let agents vote
- **Confidence weighting**: Weight contributions by self-reported confidence
- **Research swarms**: Multi-stage pipelines with question generation, research, and synthesis

Swarms trade cost for coverage. Use them when multiple perspectives add value.